In [ ]:
# @title Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Classical Lustre & Heptagon Examples, Clock Synchronization, Bidirectional `post`, and Mixture of Experts

This notebook implements canonical examples from the **Lustre**, **Lucid Synchrone**, and **Heptagon** literature alongside recent reactive machine-learning examples, complete with **multi-cycle chronograms** (horizons 6 to 10) and **Stage-1 schematic graph visualizations**:

1. **Lustre & Heptagon Basics (Horizon 8)**: Rising-edge detector (`b and not (false fby b)`) and multi-rate subclock node `two(m)` from Gaudelier et al., *FDL 2026* ([arXiv:2607.21797](https://arxiv.org/abs/2607.21797), Fig. 2).
2. **UP/DOWN Mode Automaton (Horizon 10)**: Two-mode synchronous automaton with shared state memory (`last`), weak transitions (`until`), strong preemption (`unless`), and per-mode local counters.
3. **Modular ABRO Synchronous Automata (Horizon 10)**: Gérard Berry's canonical Esterel / Heptagon benchmark (`await A and B; emit O; reset on R`), factoring the 4-state product automaton into two communicating 2-state `A`- and `B`-specific automata across three modular formulations alongside the equivalent pure `reset(r)` dataflow formulation.
4. **Cruise Control Mode Automaton (Horizon 10)**: Classical 4-state automotive controller (`OFF`, `STANDBY`, `CRUISING`, `BRAKING`) with strong brake/off preemption and an internal PI controller whose integral state resets automatically on mode re-entry.
5. **Synchronization Errors & Relaxed Clock Calculus (Horizons 8–10)**:
   - *Adding all natural numbers and all odd numbers*: static `ClockError` detection (`incompatible input clocks . vs . on is_odd`), FIFO token drift in Kahn semantics, and synchronized repairs via `current` (sample-and-hold) or `when` (subclock alignment).
   - *Gaudelier et al., FDL 2026 ([arXiv:2607.21797](https://arxiv.org/abs/2607.21797), Figs. 3 & 4)*: backpropagated `simple_net` (`. on train`) vs `gated_net`, where forward gating (`. on gate`) and backward training (`. on train`) yield syntactic clocks `. on train on gate` vs `. on gate on train` that classical Lustre clock calculus rejects unless relaxed to conjunction congruence (`gate & train`).
6. **Bidirectional Reactivity with `post`: `backfill`, `fby_end`, and `batch_norm` (Horizons 6–8)**: Bounded reverse-time lookahead from Cohen et al. ([arXiv:2311.16977](https://arxiv.org/pdf/2311.16977), Figs. 16–18), reproducing the exact paper chronograms for `backfill(i, end)`, `fby_end(end, init, i)`, and two-pass online batch normalization.
7. **Sparsely-Gated Mixture of Experts (Horizon 8)**: Conditional expert routing with `when` / `merge` and time-modular stateful experts from [arXiv:2311.16977](https://arxiv.org/pdf/2311.16977) (Fig. 12 & §4.4).

In [ ]:
# @title Psyre imports and chronogram / inspection helpers

from collections.abc import Callable, Mapping, Sequence
import logging
from typing import Any
# !pip install -q psyre

# pylint: disable=g-importing-member,g-multiple-import
from psyre.analysis import (
    CausalityError,
    check_causality,
    topological_sort,
)
from psyre.automaton import (
    automaton,
    state,
    unless,
    until,
)
from psyre.builder import build_graph
from psyre.clock import (
    BASE_CLOCK,
    Clock,
    ClockAnalysis,
    ClockError,
    check_clocks,
)
from psyre.common import (
    Absent,
    buffer,
    counter,
    current,
    fby,
    ifthenelse,
    logger,
    merge,
    post,
    pre,
    prefix_sum,
    psyre_comb,
    psyre_node,
    reset,
    scan,
    switch,
    when,
    whennot,
)
from examples.gmoe import mixture_of_experts_demo
from psyre.graph import Graph
from psyre.match import (
    case,
    default,
    last,
    match,
)
from psyre.scheduler import Scheduler
from psyre.stream import Stream
from psyre.visualize import visualize_graph

logger.setLevel(logging.WARNING)


def format_cell(val: Any) -> str:
  """Formats a single chronogram value (`□` for Absent, `T`/`F` for bools)."""
  if val is Absent:
    return '□'
  if isinstance(val, bool):
    return 'T' if val else 'F'
  if val is None:
    return '⊥'
  if isinstance(val, float):
    return (
        f'{val:.2f}'.rstrip('0').rstrip('.')
        if val != int(val)
        else f'{val:.1f}'
    )
  return str(val)


def print_chronogram(
    title: str,
    traces: Mapping[str, Sequence[Any]],
    *,
    horizon: int | None = None,
    clocks: Mapping[str, Any] | None = None,
) -> None:
  """Prints an aligned multi-cycle chronogram table across cycles 0..horizon-1."""
  if not traces:
    return
  lengths = [len(seq) for seq in traces.values()]
  n_steps = horizon if horizon is not None else max(lengths, default=0)
  formatted_rows: dict[str, list[str]] = {}
  for name, seq in traces.items():
    formatted_rows[name] = [
        format_cell(seq[k]) if k < len(seq) else '·' for k in range(n_steps)
    ]

  label_map: dict[str, str] = {}
  for name in traces:
    if clocks and name in clocks:
      label_map[name] = f'{name} :: {clocks[name]}'
    else:
      label_map[name] = name

  name_w = max(len('cycle'), *(len(lbl) for lbl in label_map.values()))
  col_widths = [
      max(
          len(str(k)),
          *(len(formatted_rows[name][k]) for name in traces),
          3,
      )
      for k in range(n_steps)
  ]

  header_cells = ' | '.join(f'{k:>{col_widths[k]}}' for k in range(n_steps))
  header = f"{'cycle':<{name_w}} | {header_cells}"
  sep = '-' * len(header)
  print(f'=== {title} ===')
  print(header)
  print(sep)
  for name, row in formatted_rows.items():
    cells = ' | '.join(f'{row[k]:>{col_widths[k]}}' for k in range(n_steps))
    print(f'{label_map[name]:<{name_w}} | {cells}')
  print()


def expect_error(
    title: str, exc_type: type[Exception], fn: Callable[[], Any]
) -> None:
  """Asserts that `fn()` raises `exc_type` and prints the diagnostic."""
  try:
    fn()
  except exc_type as err:
    print(f'[{title}] {exc_type.__name__}: {err}')
    return
  raise AssertionError(f'Expected {exc_type.__name__} for {title!r}.')

## 1. Lustre & Heptagon Basics: Rising-Edge Detector & Node `two(m)`

We start with two foundational synchronous dataflow kernels:
- **Rising-edge detector**: `edge = b and not (false fby b)` detects `False -> True` transitions on a boolean stream `b`.
- **Node `two(m)`** from Gaudelier et al., *FDL 2026* ([arXiv:2607.21797](https://arxiv.org/abs/2607.21797), Fig. 2):
  ```lustre
  node two(m: bool) returns (o: int);
  var x: int;
  let
    o = 0 fby x;
    x = merge(m, (o when m) + 1, (o when not m) + 2);
  tel
  ```
  When `m` is `True`, `o` is sampled onto `. on m` and incremented by `1`; when `m` is `False`, `o` is sampled onto `. on not m` and incremented by `2`; `merge` recombines the complementary sub-clocks back onto the base clock `.`.

In [ ]:
# @title 1.1 Chronogram and clock inference for `edge` and `two(m)`


def lustre_basics_demo() -> dict[str, Stream]:
  b = Stream(
      [False, True, True, False, True, False, True, True], name='b'
  )
  m = Stream(
      [True, True, False, True, False, False, True, False], name='m'
  )
  # Rising-edge detector: edge = b and not (false fby b)
  rising_edge = b & ~fby(False, b, name='prev_b')
  rising_edge.name = 'edge'

  # Node `two(m)` from Gaudelier et al. FDL 2026 (arXiv:2607.21797, Fig. 2):
  o = fby(0, lambda: x, name='o')  # pylint: disable=undefined-variable
  o_when_m = when(m, o, name='o_when_m') + 1
  o_whennot_m = whennot(m, o, name='o_whennot_m') + 2
  x = merge(m, o_when_m, o_whennot_m, name='x')
  return {
      'b': b,
      'edge': rising_edge,
      'm': m,
      'o': o,
      'o_when_m': o_when_m,
      'o_whennot_m': o_whennot_m,
      'x': x,
  }


g_basics = build_graph(lustre_basics_demo)
clocks_basics = check_clocks(g_basics)
traces_basics = Scheduler.run_graph(g_basics, horizon=8, clocks=clocks_basics)

print_chronogram(
    '1. Lustre Rising-Edge Detector & Node two(m) (horizon=8)',
    traces_basics,
    clocks={
        k: clocks_basics.stream_clock(s) for k, s in g_basics.output.items()
    },
)
assert traces_basics['edge'] == [
    False, True, False, False, True, False, True, False
]
assert traces_basics['o'] == [0, 1, 2, 4, 5, 7, 9, 10]
assert traces_basics['x'] == [1, 2, 4, 5, 7, 9, 10, 12]

In [ ]:
# @title 1.2 Stage-1 schematic visualization of `lustre_basics_demo`

visualize_graph(
    g_basics,
    title='Lustre Rising-Edge Detector & Node two(m) (FDL 2026 Fig. 2)',
    height_px=480,
)

## 2. UP/DOWN Mode Automaton

A classic **Heptagon / Lucid Synchrone mode automaton** switching between two counting modes (`UP` and `DOWN`) with shared state memory `prev_v = last(lambda: fsm.v, init=0)`:
- In state `UP`, `v = prev_v + 2`, and a mode-local counter `ticks_in_mode` counts consecutive cycles spent in `UP` starting from `1`. A **weak transition** `until(toggle | (v >= 6), 'DOWN')` switches to `DOWN` at the *next* cycle once `toggle` is asserted or `v` reaches `6`.
- In state `DOWN`, `v = prev_v - 1`, and a separate local counter counts cycles spent in `DOWN`. A **strong preemption** `unless(clear, 'UP')` immediately preempts `DOWN` *before* its body executes in the cycle where `clear` is `True`, resetting the `UP` state equations at that instant.

In [ ]:
# @title 2.1 UP/DOWN automaton chronogram


def up_down_automaton_demo() -> dict[str, Stream]:
  toggle = Stream(
      [False, False, True, False, False, True, False, False, False, False],
      name='toggle',
  )
  clear = Stream(
      [False, False, False, False, False, False, False, True, False, False],
      name='clear',
  )
  with automaton(init='UP', name='up_down') as fsm:
    prev_v = last(lambda: fsm.v, init=0, name='prev_v')
    with state('UP', unless(clear, 'UP')):
      fsm.mode = 'UP'
      fsm.v = prev_v + 2
      fsm.ticks_in_mode = counter(start=1, name='up_ticks')
      until(toggle | (fsm.v >= 6), 'DOWN')
    with state('DOWN', unless(clear, 'UP')):
      fsm.mode = 'DOWN'
      fsm.v = prev_v - 1
      fsm.ticks_in_mode = counter(start=1, name='down_ticks')
      until(toggle | (fsm.v <= 0), 'UP')

  return {
      'toggle': toggle,
      'clear': clear,
      'mode': fsm.mode,
      'v': fsm.v,
      'ticks_in_mode': fsm.ticks_in_mode,
  }


g_updown = build_graph(up_down_automaton_demo)
clocks_updown = check_clocks(g_updown)
traces_updown = Scheduler.run_graph(g_updown, horizon=10, clocks=clocks_updown)

print_chronogram('2. UP/DOWN Mode Automaton (horizon=10)', traces_updown)
assert traces_updown['mode'] == [
    'UP', 'UP', 'UP', 'DOWN', 'DOWN', 'DOWN', 'UP', 'UP', 'DOWN', 'DOWN'
]
assert traces_updown['v'] == [2, 4, 6, 5, 4, 3, 5, 7, 6, 5]

In [ ]:
# @title 2.2 Stage-1 schematic visualization of the UP/DOWN mode automaton

visualize_graph(
    g_updown,
    title='UP/DOWN Mode Automaton (Hierarchical Stage-1 Lowering)',
    height_px=560,
)

## 3. Modular ABRO Synchronous Automata

Gérard Berry's **ABRO** specification (*"Await `A` and `B` in parallel, emit `O` in the exact instant when both have occurred, and restart whenever `R` occurs with strong preemption"*) is the canonical benchmark for synchronous languages (Esterel, Lustre, Lucid Synchrone, and Heptagon).

A monolithic 4-state mode automaton (`WAIT_AB`, `WAIT_B`, `WAIT_A`, `DONE`) is simply the Cartesian product `{WAIT_A, DONE_A} × {WAIT_B, DONE_B}` of two 2-state waiters. Rather than flattening that product by hand, Psyre lets us materialize the two communicating `A`- and `B`-specific automata directly. Below we compare three modular two-automaton formulations alongside the pure `reset(r)` dataflow reference — all four produce identical cycle-by-cycle outputs (including when `r` coincides with `a` or `b`):

1. **Symmetric communicating `unless` automata (`await_a`, `await_b`)**: Two symmetric 2-state automata (`WAIT` → `DONE`) with strong preemption `unless(r, 'WAIT')`. Each emits `seen` (latched occurrence) and `just_seen` (the exact arrival pulse), combined combinatorially via `(await_a.seen & await_b.just_seen) | (await_b.seen & await_a.just_seen)` with **no extra state**.
2. **Directed communicating `unless` automata (`abro_a` → `abro_b`)**: `abro_a` tracks `a` and passes its `seen` and `just_seen` streams directly into `abro_b`, whose `WAIT_B` / `DONE_B` states compute `o` (`abro_a.seen & b` vs. `abro_a.just_seen`) with **no extra state**.
3. **Modular `with reset(r):` automata (`await_a`, `await_b`)**: Classic Lucid Synchrone / Heptagon style — two minimal 2-state `WAIT` → `DONE` automata with no per-state `unless` clauses, enclosed in a single `with reset(r):` block whose `seen_a & seen_b` conjunction feeds a rising-edge detector (`both & ~fby(False, both)`).
4. **Pure `reset(r)` dataflow**: Two latched boolean `fby` memories `seen_a` and `seen_b` inside `with reset(r):` plus the same rising-edge detector.

In [ ]:
# @title 3.1 Modular ABRO automata and `reset(r)` dataflow chronogram

_ABRO_A = [False, True, False, False, True, False, True, False, True, True]
_ABRO_B = [False, False, True, False, True, False, False, False, False, True]
_ABRO_R = [False, False, False, True, False, False, False, True, False, False]


def _abro_inputs() -> tuple[Stream, Stream, Stream]:
  return (
      Stream(_ABRO_A, name='a'),
      Stream(_ABRO_B, name='b'),
      Stream(_ABRO_R, name='r'),
  )


def _await_unless_reset(sig: Stream, r: Stream, name: str):
  """2-state waiter automaton with strong preemption `unless(r, 'WAIT')`."""
  with automaton(init='WAIT', name=name) as fsm:
    with state('WAIT', unless(r, 'WAIT')):
      fsm.state_tag = 'WAIT'
      fsm.seen = sig
      fsm.just_seen = sig
      until(sig, 'DONE')
    with state('DONE', unless(r, 'WAIT')):
      fsm.state_tag = 'DONE'
      fsm.seen = True
      fsm.just_seen = False
  return fsm


def _await_simple(sig: Stream, name: str):
  """Minimal 2-state waiter automaton reset externally via `with reset(r):`."""
  with automaton(init='WAIT', name=name) as fsm:
    with state('WAIT'):
      fsm.state_tag = 'WAIT'
      fsm.seen = sig
      until(sig, 'DONE')
    with state('DONE'):
      fsm.state_tag = 'DONE'
      fsm.seen = True
  return fsm


# 1. Symmetric communicating 2-state automata with strong `unless(r, 'WAIT')`:
def abro_symmetric_demo() -> dict[str, Stream]:
  a, b, r = _abro_inputs()
  fsm_a = _await_unless_reset(a, r, 'await_a')
  fsm_b = _await_unless_reset(b, r, 'await_b')
  o = (fsm_a.seen & fsm_b.just_seen) | (fsm_b.seen & fsm_a.just_seen)
  return {
      'a': a,
      'b': b,
      'r': r,
      'state_a': fsm_a.state_tag,
      'state_b': fsm_b.state_tag,
      'o_symmetric': o,
  }


# 2. Directed communicating 2-state automata (`abro_a` -> `abro_b`):
def abro_directed_demo() -> dict[str, Stream]:
  a, b, r = _abro_inputs()
  with automaton(init='WAIT_A', name='abro_a') as fsm_a:
    with state('WAIT_A', unless(r, 'WAIT_A')):
      fsm_a.state_tag = 'WAIT_A'
      fsm_a.seen = a
      fsm_a.just_seen = a
      until(a, 'DONE_A')
    with state('DONE_A', unless(r, 'WAIT_A')):
      fsm_a.state_tag = 'DONE_A'
      fsm_a.seen = True
      fsm_a.just_seen = False

  with automaton(init='WAIT_B', name='abro_b') as fsm_b:
    with state('WAIT_B', unless(r, 'WAIT_B')):
      fsm_b.state_tag = 'WAIT_B'
      fsm_b.o = fsm_a.seen & b
      until(b, 'DONE_B')
    with state('DONE_B', unless(r, 'WAIT_B')):
      fsm_b.state_tag = 'DONE_B'
      fsm_b.o = fsm_a.just_seen

  return {
      'a': a,
      'b': b,
      'r': r,
      'state_a': fsm_a.state_tag,
      'state_b': fsm_b.state_tag,
      'o_directed': fsm_b.o,
  }


# 3. Modular `with reset(r):` wrapping two minimal 2-state automata + edge:
def abro_reset_automata_demo() -> dict[str, Stream]:
  a, b, r = _abro_inputs()
  with reset(r):
    fsm_a = _await_simple(a, 'await_a')
    fsm_b = _await_simple(b, 'await_b')
    both = fsm_a.seen & fsm_b.seen
    o = both & ~fby(False, both, name='prev_both')
  return {
      'a': a,
      'b': b,
      'r': r,
      'state_a': fsm_a.state_tag,
      'state_b': fsm_b.state_tag,
      'o_reset_fsm': o,
  }


# 4. Equivalent pure synchronous dataflow equation using modular `reset(r)`:
def abro_dataflow_demo() -> dict[str, Stream]:
  a, b, r = _abro_inputs()
  with reset(r):
    seen_a = fby(False, lambda: seen_a, name='seen_a') | a
    seen_b = fby(False, lambda: seen_b, name='seen_b') | b
    both = seen_a & seen_b
    o = both & ~fby(False, both, name='prev_both')
  return {'o_dataflow': o}


g_abro_sym = build_graph(abro_symmetric_demo)
clocks_abro_sym = check_clocks(g_abro_sym)
traces_abro_sym = Scheduler.run_graph(
    g_abro_sym, horizon=10, clocks=clocks_abro_sym
)

g_abro_dir = build_graph(abro_directed_demo)
clocks_abro_dir = check_clocks(g_abro_dir)
traces_abro_dir = Scheduler.run_graph(
    g_abro_dir, horizon=10, clocks=clocks_abro_dir
)

g_abro_rst = build_graph(abro_reset_automata_demo)
clocks_abro_rst = check_clocks(g_abro_rst)
traces_abro_rst = Scheduler.run_graph(
    g_abro_rst, horizon=10, clocks=clocks_abro_rst
)

g_abro_df = build_graph(abro_dataflow_demo)
clocks_abro_df = check_clocks(g_abro_df)
traces_abro_df = Scheduler.run_graph(
    g_abro_df, horizon=10, clocks=clocks_abro_df
)

traces_abro = {
    'a': traces_abro_sym['a'],
    'b': traces_abro_sym['b'],
    'r': traces_abro_sym['r'],
    'state_a': traces_abro_sym['state_a'],
    'state_b': traces_abro_sym['state_b'],
    'o_symmetric': traces_abro_sym['o_symmetric'],
    'o_directed': traces_abro_dir['o_directed'],
    'o_reset_fsm': traces_abro_rst['o_reset_fsm'],
    'o_dataflow': traces_abro_df['o_dataflow'],
}

print_chronogram(
    '3. Modular ABRO Automata & Reset Dataflow Equivalence (horizon=10)',
    traces_abro,
)
expected_o = [False, False, True, False, True, False, False, False, False, True]
assert traces_abro['o_symmetric'] == expected_o
assert traces_abro['o_directed'] == expected_o
assert traces_abro['o_reset_fsm'] == expected_o
assert traces_abro['o_dataflow'] == expected_o
assert traces_abro_sym['state_a'] == traces_abro_rst['state_a']
assert traces_abro_sym['state_b'] == traces_abro_rst['state_b']

In [ ]:
# @title 3.2 Stage-1 schematic: Symmetric communicating ABRO automata

visualize_graph(
    g_abro_sym,
    title='ABRO (1): Symmetric Communicating Automata (unless strong reset)',
    height_px=560,
)

In [ ]:
# @title 3.3 Stage-1 schematic: Direct-communication ABRO automata (`abro_a` -> `abro_b`)

visualize_graph(
    g_abro_dir,
    title='ABRO (2): Directed Communicating Automata (abro_a -> abro_b)',
    height_px=560,
)

In [ ]:
# @title 3.4 Stage-1 schematic: Modular `with reset(r):` ABRO automata

visualize_graph(
    g_abro_rst,
    title='ABRO (3): Modular reset(r) Automata + Rising-Edge Detector',
    height_px=560,
)

## 4. Cruise Control Mode Automaton

The **Cruise Control** mode automaton is a standard Lustre / Heptagon / SCADE case study mixing hierarchical discrete mode transitions with continuous PI control:
- **`OFF`**: Target speed `v_target = 0.0` and `throttle = 0.0`. Weak transition `until(on_off, 'STANDBY')` powers the system on at the next cycle.
- **`STANDBY`**: Captures `v_target = speed` when `set_speed` is pressed and transitions via `until(set_speed & ~brake, 'CRUISING')`.
- **`CRUISING`**: Strongly preempted by `unless(on_off, 'OFF')` or `unless(brake, 'BRAKING')` (so pressing the brake cuts throttle to `0.0` in the *same* cycle). While cruising, `v_target` can be bumped by `quick_accel` (`+2.0`), and a local PI regulator (`prefix_sum(err)`) computes `throttle`. Because `prefix_sum` lives inside `state('CRUISING')`, re-entering `CRUISING` after braking automatically resets the integral accumulator.
- **`BRAKING`**: Holds the previous target speed (`prev_target = last(lambda: cc.v_target, init=0.0)`) with `throttle = 0.0` until `~brake & resume` re-engages `CRUISING`.

In [ ]:
# @title 4.1 Cruise control mode automaton chronogram


def cruise_control_demo() -> dict[str, Stream]:
  on_off = Stream(
      [True, False, False, False, False, False, False, False, False, True],
      name='on_off',
  )
  set_speed = Stream(
      [False, True, False, False, False, False, False, False, False, False],
      name='set_speed',
  )
  brake = Stream(
      [False, False, False, False, False, True, False, False, False, False],
      name='brake',
  )
  resume = Stream(
      [False, False, False, False, False, False, False, True, False, False],
      name='resume',
  )
  quick_accel = Stream(
      [False, False, False, True, False, False, False, False, False, False],
      name='quick_accel',
  )
  speed = Stream(
      [90.0, 100.0, 96.0, 98.0, 101.0, 95.0, 90.0, 92.0, 97.0, 100.0],
      name='speed',
  )

  kp, ki = 1.5, 0.5
  with automaton(init='OFF', name='cruise_ctrl') as cc:
    prev_target = last(lambda: cc.v_target, init=0.0, name='prev_target')
    with state('OFF'):
      cc.mode = 'OFF'
      cc.v_target = 0.0
      cc.throttle = 0.0
      until(on_off, 'STANDBY')
    with state('STANDBY', unless(on_off, 'OFF')):
      cc.mode = 'STANDBY'
      cc.v_target = ifthenelse(set_speed, speed, prev_target)
      cc.throttle = 0.0
      until(set_speed & ~brake, 'CRUISING')
    with state('CRUISING', unless(on_off, 'OFF'), unless(brake, 'BRAKING')):
      cc.mode = 'CRUISING'
      cc.v_target = ifthenelse(
          set_speed,
          speed,
          ifthenelse(quick_accel, prev_target + 2.0, prev_target),
      )
      err = cc.v_target - speed
      integ = prefix_sum(err, start=0.0, name='pi_integ')
      cc.throttle = (kp * err + ki * integ).max(0.0)
    with state('BRAKING', unless(on_off, 'OFF')):
      cc.mode = 'BRAKING'
      cc.v_target = prev_target
      cc.throttle = 0.0
      until(~brake & resume, 'CRUISING')

  return {
      'on_off': on_off,
      'set_speed': set_speed,
      'quick_accel': quick_accel,
      'brake': brake,
      'resume': resume,
      'speed': speed,
      'mode': cc.mode,
      'v_target': cc.v_target,
      'throttle': cc.throttle,
  }


g_cc = build_graph(cruise_control_demo)
clocks_cc = check_clocks(g_cc)
traces_cc = Scheduler.run_graph(g_cc, horizon=10, clocks=clocks_cc)

print_chronogram('4. Cruise Control Mode Automaton (horizon=10)', traces_cc)
assert traces_cc['mode'] == [
    'OFF',
    'STANDBY',
    'CRUISING',
    'CRUISING',
    'CRUISING',
    'BRAKING',
    'BRAKING',
    'BRAKING',
    'CRUISING',
    'OFF',
]

In [ ]:
# @title 4.2 Stage-1 schematic visualization of the Cruise Control automaton

visualize_graph(
    g_cc,
    title='Cruise Control 4-State Mode Automaton (OFF / STANDBY / CRUISING / BRAKING)',
    height_px=640,
)

## 5. Synchronization Errors & Relaxed Clock Calculus

### 5.1 Adding All Natural Numbers and All Odd Numbers

A classic Lustre / Lucid Synchrone clock-calculus example attempts to add the stream of all natural numbers `nat = 0, 1, 2, 3, ...` (on the base clock `.`) to the subsampled stream of odd numbers `odd = nat when is_odd` (on the subclock `. on is_odd`):
- **Static clock check (`check_clocks`)**: Immediately rejects `nat + odd` with `ClockError: incompatible input clocks . vs . on is_odd`.
- **Unclocked synchronous vs Kahn execution**: Without static clock checking, `Semantics.SYNC` emits `Absent` (`□`) at every even cycle on a stream supposed to live on the base clock, whereas `Semantics.KAHN` pairs the $k$-th natural number with the $(2k+1)$-th odd number (`0+1=1, 1+3=4, 2+5=7, ...`), causing **FIFO token drift** and unbounded buffer accumulation on `nat`.
- **Well-clocked repairs**:
  1. **Base-clock sample-and-hold (`current`)**: `held_odd = current(is_odd, nat, default=0)` holds the latest odd number on `.` so `nat + held_odd` runs every cycle on `.`.
  2. **Subclock alignment (`when`)**: `when(is_odd, nat) + odd` downsamples `nat` onto `. on is_odd` before adding.

In [ ]:
# @title 5.1a Static `ClockError`, token drift, and synchronized repairs for `nat + odd`


def bad_nat_plus_odd() -> Stream:
  nat = counter(name='nat')
  is_odd = (nat % 2) == 1
  is_odd.name = 'is_odd'
  odd = when(is_odd, nat, name='odd')
  return nat + odd


g_bad_nat_odd = build_graph(bad_nat_plus_odd)
expect_error(
    '5.1a nat + odd [SYNC ClockError]',
    ClockError,
    lambda: check_clocks(g_bad_nat_odd),
)

# Without clock checking in SYNC, `nat + odd` produces `□` (Absent) on even cycles:
sync_unclocked = Scheduler.run(
    bad_nat_plus_odd,
    horizon=10,
    clock_calculus=False,
)
print(
    '[Unclocked SYNC trace of nat + odd]:',
    [format_cell(v) for v in sync_unclocked],
)
print()


def fixed_nat_plus_odd() -> dict[str, Stream]:
  nat = counter(name='nat')
  is_odd = (nat % 2) == 1
  is_odd.name = 'is_odd'
  odd = when(is_odd, nat, name='odd')
  # Repair 1 (base clock `.`): sample-and-hold via `current`
  held_odd = current(is_odd, nat, default=0, name='held_odd')
  sum_base = nat + held_odd
  sum_base.name = 'sum_base'
  # Repair 2 (subclock `. on is_odd`): downsample `nat` onto `. on is_odd`
  sum_sub = when(is_odd, nat, name='nat_on_odd') + odd
  sum_sub.name = 'sum_sub'
  return {
      'nat': nat,
      'is_odd': is_odd,
      'odd': odd,
      'held_odd': held_odd,
      'sum_base': sum_base,
      'sum_sub': sum_sub,
  }


g_nat_odd = build_graph(fixed_nat_plus_odd)
clocks_nat_odd = check_clocks(g_nat_odd)
traces_nat_odd = Scheduler.run_graph(
    g_nat_odd, horizon=10, clocks=clocks_nat_odd
)

print_chronogram(
    '5.1b Synchronized Natural + Odd Repairs in SYNC (horizon=10)',
    traces_nat_odd,
    clocks={
        k: clocks_nat_odd.stream_clock(s) for k, s in g_nat_odd.output.items()
    },
)

In [ ]:
# @title 5.1b Schematic visualization of synchronized `nat + odd` repairs

visualize_graph(
    g_nat_odd,
    title='Synchronized Natural + Odd Repairs (current onto . vs when onto . on is_odd)',
    height_px=520,
)

### 5.2 Gaudelier et al., FDL 2026 ([arXiv:2607.21797](https://arxiv.org/abs/2607.21797), Figs. 3 & 4): `simple_net` vs `gated_net` Backpropagation

Gaudelier et al. (*"Towards a Relaxed Clock Calculus for Synchronous Backpropagation"*, FDL 2026, [arXiv:2607.21797](https://arxiv.org/abs/2607.21797)) show how automatic reverse-mode differentiation of multi-rate synchronous networks exposes a limitation of classical syntactic clock calculus:

- **Figure 3 (`simple_net`)**: A linear layer `x = i * k` whose backward pass is gated by a single boolean stream `train`. All backward streams (`it`, `xt`, `gx`, `gw`, `wr`) share the single subclock `. on train`, which classical Lustre clock calculus accepts directly.
- **Figure 4 (`gated_net`)**: The forward pass gates the input by `gate` (`ic = i when gate`, `a = kc * ic`, `x = merge(gate, a, 0.0)`), while the backward pass gates training by `train` (`gx` on `. on train`).
  - Differentiating `x = merge(gate, a, 0.0)` samples `gx` by `gate when train`, giving `ga` the clock **`. on train on gate`**.
  - Differentiating `a = kc * ic` requires sampling the forward activation `ic` (already on `. on gate`) by `train when gate`, giving `it` the clock **`. on gate on train`**.
  - Multiplying `gw = ga * it` therefore combines **`. on train on gate`** and **`. on gate on train`**. Classical syntactic clock calculus raises `ClockError`, even though both clocks are active at the exact same instants (`gate & train`)!

In [ ]:
# @title 5.2a FDL 2026 Fig. 3 (`simple_net`) and Fig. 4 (`gated_net`) chronograms


def fdl2026_simple_net() -> dict[str, Stream]:
  """Simple backpropagated network from Gaudelier et al. FDL 2026, Fig. 3."""
  lr = 0.1
  train = Stream(
      [False, True, True, False, True, True, False, True], name='train'
  )
  inp = Stream([1.0, 2.0, 1.0, 3.0, 2.0, 1.0, 2.0, 2.0], name='i')
  wi = Stream(0.5, name='wi')
  y = when(
      train,
      Stream([0.0, 4.0, 2.0, 0.0, 4.0, 2.0, 0.0, 4.0]),
      name='y',
  )

  x = inp * (lambda: net.k)  # pylint: disable=undefined-variable
  x.name = 'x'
  it = when(train, inp, name='it')
  xt = when(train, x, name='xt')
  gx = 2.0 * (xt - y)
  gx.name = 'gx'
  gw = gx * it
  gw.name = 'gw'

  with match(train) as net:
    with case(True):
      wr_state = fby(wi, lambda: wr_state - lr * gw, name='wr_state')
      net.k = wr_state
    with case(False):
      net.k = wi

  wr = when(train, net.k, name='wr')
  return {
      'train': train,
      'i': inp,
      'k': net.k,
      'x': x,
      'y': y,
      'gx': gx,
      'wr': wr,
  }


g_simple_net = build_graph(fdl2026_simple_net)
clocks_simple_net = check_clocks(g_simple_net)
traces_simple_net = Scheduler.run_graph(
    g_simple_net, horizon=8, clocks=clocks_simple_net
)
print_chronogram(
    '5.2a FDL 2026 Fig. 3: simple_net Backpropagation (horizon=8)',
    traces_simple_net,
    clocks={
        k: clocks_simple_net.stream_clock(s)
        for k, s in g_simple_net.output.items()
    },
)


def fdl2026_gated_backprop_syntactic_conflict() -> dict[str, Stream]:
  """Gated layer backpropagation (FDL 2026 Fig. 4): `. on train on gate` vs `. on gate on train`."""
  inp = Stream([1.0, 2.0, 3.0, 4.0, 1.5, 2.5, 3.5, 0.5], name='i')
  gate = Stream(
      [True, False, True, True, False, True, False, True], name='gate'
  )
  train = Stream(
      [False, False, True, True, True, False, True, True], name='train'
  )
  target = when(
      train, Stream([0.0, 0.0, 4.0, 6.0, 1.0, 0.0, 0.0, 2.0]), name='y'
  )

  # Forward gated path: sample input by `gate` (. on gate)
  ic = when(gate, inp, name='ic')
  kc = Stream(1.5, name='kc')
  a = kc * ic
  x = merge(gate, a, 0.0, name='x')

  # Backward path gated by `train`: gradient `gx` has clock `. on train`
  xt = when(train, x, name='xt')
  gx = 2.0 * (xt - target)
  gx.name = 'gx'

  # Differentiating `x = merge(gate, a, 0.0)` samples `gx` by `gate when train`:
  # Clock of `ga` is `. on train on gate`
  gate_train = when(train, gate, name='gate_train')
  ga = when(gate_train, gx, name='ga')

  # Sampling forward activation `ic` (. on gate) by `train when gate`:
  # Clock of `it` is `. on gate on train`
  train_gate = when(gate, train, name='train_gate')
  it = when(train_gate, ic, name='it')

  # Weight gradient multiplies `ga` (. on train on gate) and `it` (. on gate on train):
  gw = ga * it
  gw.name = 'gw'
  return {
      'i': inp,
      'gate': gate,
      'train': train,
      'x': x,
      'ga': ga,
      'it': it,
      'gw': gw,
  }


g_gated_conflict = build_graph(
    fdl2026_gated_backprop_syntactic_conflict
)
expect_error(
    '5.2b FDL 2026 Fig. 4 gated_net [Syntactic ClockError]',
    ClockError,
    lambda: check_clocks(g_gated_conflict),
)
print()

# Under relaxed clock congruence (`clock_calculus=False`), `. on train on gate` and
# `. on gate on train` are synchronous whenever `gate & train` holds:
traces_fdl_relaxed = Scheduler.run_graph(
    g_gated_conflict, horizon=8, clocks=None
)
print_chronogram(
    '5.2c FDL 2026 Fig. 4: gated_net Under Relaxed Clock Congruence (horizon=8)',
    traces_fdl_relaxed,
)


def fdl2026_gated_backprop_canonical() -> dict[str, Stream]:
  """Gated layer backpropagation with canonical conjunction clock `active = gate & train`."""
  inp = Stream([1.0, 2.0, 3.0, 4.0, 1.5, 2.5, 3.5, 0.5], name='i')
  gate = Stream(
      [True, False, True, True, False, True, False, True], name='gate'
  )
  train = Stream(
      [False, False, True, True, True, False, True, True], name='train'
  )
  target = when(
      train, Stream([0.0, 0.0, 4.0, 6.0, 1.0, 0.0, 0.0, 2.0]), name='y'
  )

  ic = when(gate, inp, name='ic')
  kc = Stream(1.5, name='kc')
  a = kc * ic
  x = merge(gate, a, 0.0, name='x')

  xt = when(train, x, name='xt')
  gx = 2.0 * (xt - target)
  gx.name = 'gx'

  # Canonicalizing the conjunction `active = gate & train` aligns both streams on `. on active`:
  active = gate & train
  active.name = 'active'
  ga = when(active, merge(train, gx, 0.0), name='ga')
  it = when(active, merge(gate, ic, 0.0), name='it')
  gw = ga * it
  gw.name = 'gw'
  return {
      'i': inp,
      'gate': gate,
      'train': train,
      'active': active,
      'x': x,
      'ga': ga,
      'it': it,
      'gw': gw,
  }


g_gated_can = build_graph(fdl2026_gated_backprop_canonical)
clocks_gated_can = check_clocks(g_gated_can)
traces_fdl_can = Scheduler.run_graph(
    g_gated_can, horizon=8, clocks=clocks_gated_can
)
assert traces_fdl_can['gw'] == traces_fdl_relaxed['gw']

In [ ]:
# @title 5.2b Stage-1 schematic visualization of FDL 2026 Fig. 3 (`simple_net`)

visualize_graph(
    g_simple_net,
    title='FDL 2026 Fig. 3: simple_net Backpropagation (. on train)',
    height_px=520,
)

In [ ]:
# @title 5.2c Stage-1 schematic visualization of FDL 2026 Fig. 4 (`gated_net` with conjunction clock `active = gate & train`)

visualize_graph(
    g_gated_can,
    title='FDL 2026 Fig. 4: gated_net Backpropagation (. on active where active = gate & train)',
    height_px=540,
)

## 6. Bidirectional Reactivity with `post`: `backfill`, `fby_end`, and `batch_norm`

In *"Synchronous Programming for Neural Network Layers"* ([arXiv:2311.16977](https://arxiv.org/pdf/2311.16977), §5.1, Figs. 16–18), Cohen et al. introduce the reverse-time lookahead primitive **`post(e)`** (the temporal dual of `pre(e)`), together with **`backfill`** and **`fby_end`**:

- **`backfill(i, bp)`** (Figure 16): Broadcasts the value of `i` at each boundary instant where `bp` is `True` *backwards in time* to earlier instants of the same window:
  $$o = \text{merge}(bp,\; i \text{ when } bp,\; (\text{post } o) \text{ when not } bp)$$
  Because each Psyre node coroutine steps forward in time ($k = 0, 1, \dots$), a direct self-loop through `post(lambda: o)` would deadlock at $k = 0$ (waiting for the same `merge` coroutine to reach $k = 1$ before it has emitted $k = 0$) and is flagged as a causality cycle by Stage 1 (`analysis.find_instantaneous_cycle`). Unrolling the recurrence across `max_span` pipeline stages turns each `post` step into a distinct upstream node coroutine that can run ahead in time.
- **`fby_end(end, init, i)`** (Figures 17–18): Piecewise forward delay `fby(init, i)` that resets at the cycle *immediately following* `end = True` (`with reset(fby(False, end)): fby(init, i)`).
- **`batch_norm` / `mean_var`** (Figure 17): Combines two forward `fby_end` accumulations (`cnt`, `sum`, `var_sum`) with two `backfill` passes (`mean` and `var`) so that every element in a mini-batch delimited by `batch_end` is normalized using the complete mini-batch's final mean and variance.

In [ ]:
# @title 6.1 Exact paper chronograms for `backfill` (Fig. 16), `fby_end` (Fig. 18), and `batch_norm` (Fig. 17)


@psyre_node('backfill')
def backfill(i: Stream, bp: Stream, *, max_span: int = 2) -> Stream:
  """Bounded broadcast towards the past using `post` (arXiv:2311.16977 Fig. 16)."""
  o = i
  for step in range(max_span):
    o = merge(bp, when(bp, i), whennot(bp, post(o)), name=f'o_{step}')
  return o


@psyre_node('fby_end')
def fby_end(end: Stream, init: Any, i: Any) -> Stream:
  """Piecewise forward recurrence resetting after `end` (arXiv:2311.16977 Fig. 17)."""
  with reset(fby(False, end, name='rst_after_end')):
    return fby(init, i, name='fby_end')


def backfill_and_fby_end_paper_demo() -> dict[str, Stream]:
  # Exact input sequences from Figures 16 and 18 of arXiv:2311.16977:
  end = Stream([False, False, True, True, False, True], name='end')
  i = Stream([2, 6, 8, 5, 1, 4], name='i')
  init = Stream([1, 0, 9, 3, 5, 8], name='init')
  o_backfill = backfill(i, end)
  o_fby_end = fby_end(end, init, i)
  return {
      'end': end,
      'i': i,
      'backfill(i, end)': o_backfill,
      'init': init,
      'fby_end(end, init, i)': o_fby_end,
  }


g_bf = build_graph(backfill_and_fby_end_paper_demo)
clocks_bf = check_clocks(g_bf)
traces_bf = Scheduler.run_graph(g_bf, horizon=6, clocks=clocks_bf)

print_chronogram(
    '6.1 Paper Exact Chronograms: backfill (Fig. 16) & fby_end (Fig. 18) (horizon=6)',
    traces_bf,
)
assert traces_bf['backfill(i, end)'] == [8, 8, 8, 5, 4, 4]
assert traces_bf['fby_end(end, init, i)'] == [1, 2, 6, 3, 5, 1]


def batch_norm_demo() -> dict[str, Stream]:
  """Batch normalization with two `backfill` passes (arXiv:2311.16977 Fig. 17)."""
  batch_end = Stream(
      [False, False, True, True, False, True, False, True], name='batch_end'
  )
  x = Stream([2.0, 5.0, 8.0, 4.0, 1.0, 5.0, 10.0, 14.0], name='x')
  cnt = 1 + fby_end(batch_end, 0, lambda: cnt)
  cnt.name = 'cnt'
  batch_sum = x + fby_end(batch_end, 0.0, lambda: batch_sum)
  batch_sum.name = 'sum'
  mean = backfill(batch_sum / cnt, batch_end, name='mean')
  var_sum = (x - mean) * (x - mean) + fby_end(batch_end, 0.0, lambda: var_sum)
  var_sum.name = 'var_sum'
  var = backfill(var_sum / cnt, batch_end, name='var')
  y = (x - mean) / ((var + 1e-5) ** 0.5)
  y.name = 'y_norm'
  return {
      'batch_end': batch_end,
      'x': x,
      'cnt': cnt,
      'sum': batch_sum,
      'mean': mean,
      'var_sum': var_sum,
      'var': var,
      'y_norm': y,
  }


g_bn = build_graph(batch_norm_demo)
clocks_bn = check_clocks(g_bn)
traces_bn = Scheduler.run_graph(g_bn, horizon=8, clocks=clocks_bn)

print_chronogram(
    '6.2 Batch Normalization via fby_end + backfill (Fig. 17, horizon=8)',
    traces_bn,
)
assert traces_bn['mean'] == [5.0, 5.0, 5.0, 4.0, 3.0, 3.0, 12.0, 12.0]
assert traces_bn['var'] == [6.0, 6.0, 6.0, 0.0, 4.0, 4.0, 4.0, 4.0]

In [ ]:
# @title 6.2 Stage-1 schematic visualization of `backfill` (Fig. 16) and `fby_end` (Fig. 18)

visualize_graph(
    g_bf,
    title='Bidirectional post: backfill (Fig. 16) & fby_end (Fig. 18)',
    height_px=520,
)

In [ ]:
# @title 6.3 Stage-1 schematic visualization of `batch_norm` (Fig. 17)

visualize_graph(
    g_bn,
    title='Batch Normalization with Two Backfill Passes (arXiv:2311.16977 Fig. 17)',
    height_px=580,
)

## 7. Sparsely-Gated Mixture of Experts

Following Figure 12 and §4.4 (*Time modularity*) of [arXiv:2311.16977](https://arxiv.org/pdf/2311.16977), a **sparsely-gated Mixture of Experts (MoE)** uses boolean clock conditions `c_k = (g_k != 0.0)` to activate only the selected experts at each cycle:
```lustre
node mixture_of_experts(x) returns (o)
  let
    g1, g2, g3 = gating_network(x);
    c1 = (g1 <> 0.0);  c2 = (g2 <> 0.0);  c3 = (g3 <> 0.0);
    o1 = expert1(x when c1);
    o2 = expert2(x when c2);
    o3 = expert3(x when c3);
    o = g1 * merge(c1, o1, 0.0)
      + g2 * merge(c2, o2, 0.0)
      + g3 * merge(c3, o3, 0.0);
  tel
```
Because each `expert_k` runs on the subclock `. on c_k`:
1. **Work efficiency**: Unselected experts (`g_k == 0.0`) are not evaluated (`□` in the chronogram).
2. **Time modularity for stateful/recurrent experts**: If an expert contains internal state (`fby` / `prefix_sum` inside `with match(c1):`), its state advances *only* at the instants where that expert is selected, remaining frozen while inactive.

In [ ]:
# @title 7.1 Sparsely-gated Mixture of Experts chronogram (from `examples/gmoe.py`)

g_moe = build_graph(mixture_of_experts_demo)
clocks_moe = check_clocks(g_moe)
traces_moe = Scheduler.run_graph(g_moe, horizon=8, clocks=clocks_moe)

print_chronogram(
    '7. Sparsely-Gated Mixture of Experts (arXiv:2311.16977 Fig. 12, horizon=8)',
    traces_moe,
    clocks={
        k: clocks_moe.stream_clock(s) for k, s in g_moe.output.items()
    },
)

In [ ]:
# @title 7.2 Stage-1 schematic visualization of the Sparsely-Gated Mixture of Experts

visualize_graph(
    g_moe,
    title='Sparsely-Gated Mixture of Experts (arXiv:2311.16977 Fig. 12)',
    height_px=580,
)